# 00 — Smoke Test

Read the OHLCV and FRED Parquet layers via Spark, verify shape/schema/dates, and audit data quality (coverage, nulls, halts, outliers).

In [1]:
import sys
sys.path.insert(0, '..')
from src.spark_session import get_spark

spark = get_spark('smoke_test')
spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/02 17:11:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## OHLCV

In [2]:
ohlcv = spark.read.parquet('../data/parquet/ohlcv')
print('total rows:', ohlcv.count())
print('distinct tickers:', ohlcv.select('ticker').distinct().count())

total rows: 1940547


distinct tickers: 503


In [3]:
ohlcv.printSchema()

root
 |-- date: date (nullable = true)
 |-- open: double (nullable = true)
 |-- high: double (nullable = true)
 |-- low: double (nullable = true)
 |-- close: double (nullable = true)
 |-- adj_close: double (nullable = true)
 |-- volume: long (nullable = true)
 |-- ticker: string (nullable = true)



In [4]:
from pyspark.sql import functions as F
ohlcv.agg(F.min('date').alias('min_date'), F.max('date').alias('max_date')).show()

+----------+----------+
|  min_date|  max_date|
+----------+----------+
|2010-01-04|2026-04-17|
+----------+----------+



In [5]:
ohlcv.orderBy('ticker','date').show(5, truncate=False)

+----------+------------------+------------------+------------------+------------------+------------------+-------+------+
|date      |open              |high              |low               |close             |adj_close         |volume |ticker|
+----------+------------------+------------------+------------------+------------------+------------------+-------+------+
|2010-01-04|22.45350456237793 |22.625179290771484|22.26752471923828 |22.389127731323242|19.810985565185547|3815561|A     |
|2010-01-05|22.324750900268555|22.3319034576416  |22.00286102294922 |22.145923614501953|19.595775604248047|4186031|A     |
|2010-01-06|22.06723976135254 |22.174535751342773|22.00286102294922 |22.06723976135254 |19.52616310119629 |3243779|A     |
|2010-01-07|22.017166137695312|22.045780181884766|21.81688117980957 |22.03862762451172 |19.50084686279297 |3095172|A     |
|2010-01-08|21.917024612426758|22.06723976135254 |21.745351791381836|22.03147315979004 |19.49452018737793 |3733918|A     |
+----------+----

## FRED macro

In [6]:
fred = spark.read.parquet('../data/parquet/fred_macro')
print('rows:', fred.count())
fred.printSchema()
fred.agg(F.min('date').alias('min_date'), F.max('date').alias('max_date')).show()
fred.orderBy('date').show(5, truncate=False)

rows: 5954
root
 |-- date: date (nullable = true)
 |-- VIXCLS: double (nullable = true)
 |-- DGS10: double (nullable = true)
 |-- FEDFUNDS: double (nullable = true)
 |-- CPIAUCSL: double (nullable = true)
 |-- UNRATE: double (nullable = true)

+----------+----------+
|  min_date|  max_date|
+----------+----------+
|2010-01-01|2026-04-20|
+----------+----------+

+----------+------+-----+--------+--------+------+
|date      |VIXCLS|DGS10|FEDFUNDS|CPIAUCSL|UNRATE|
+----------+------+-----+--------+--------+------+
|2010-01-01|NULL  |NULL |0.11    |217.488 |9.8   |
|2010-01-02|NULL  |NULL |0.11    |217.488 |9.8   |
|2010-01-03|NULL  |NULL |0.11    |217.488 |9.8   |
|2010-01-04|20.04 |3.85 |0.11    |217.488 |9.8   |
|2010-01-05|19.35 |3.77 |0.11    |217.488 |9.8   |
+----------+------+-----+--------+--------+------+
only showing top 5 rows



## Per-ticker coverage

How many trading days each ticker has. Full 16-year coverage ≈ 4,032 days; anything well below that is a recent IPO / re-listing.

In [7]:
coverage = (ohlcv.groupBy('ticker')
            .agg(F.count('*').alias('n_days'),
                 F.min('date').alias('first_date'),
                 F.max('date').alias('last_date'))
            .toPandas())
full = (coverage['n_days'] >= 4000).sum()
partial = len(coverage) - full
print(f'Tickers with full coverage (>=4000 days): {full} / {len(coverage)}')
print(f'Partial-coverage tickers (recent IPOs / new constituents): {partial}')
print(f'Mean days per ticker: {coverage["n_days"].mean():.0f}')
print('\n5 most-partial tickers:')
print(coverage.sort_values('n_days').head(5).to_string(index=False))

Tickers with full coverage (>=4000 days): 426 / 503
Partial-coverage tickers (recent IPOs / new constituents): 77
Mean days per ticker: 3858

5 most-partial tickers:
ticker  n_days first_date  last_date
     Q     119 2025-10-27 2026-04-17
  SNDK     295 2025-02-13 2026-04-17
   GEV     516 2024-03-27 2026-04-17
  SOLV     517 2024-03-26 2026-04-17
  VLTO     636 2023-10-04 2026-04-17


## Per-column null census (OHLCV)

Distributed `null` count for each column across all 1.94M rows.

In [8]:
total = ohlcv.count()
null_row = (ohlcv.select([F.sum(F.col(c).isNull().cast('int')).alias(c) for c in ohlcv.columns])
            .toPandas().T.rename(columns={0: 'nulls'}))
null_row['pct'] = (null_row['nulls'] / total * 100).round(4)
print(f'Total OHLCV rows: {total:,}')
print(null_row.to_string())

Total OHLCV rows: 1,940,547
           nulls  pct
date           0  0.0
open           0  0.0
high           0  0.0
low            0  0.0
close          0  0.0
adj_close      0  0.0
volume         0  0.0
ticker         0  0.0


## Volume = 0 days (halts / suspensions)

Halt or single-stock suspension days. Kept in the panel (price still meaningful) but flagged via `volume = 0`.

In [9]:
zero_vol = ohlcv.filter(F.col('volume') == 0).count()
print(f'Rows with volume == 0: {zero_vol:,} / {total:,} ({zero_vol/total*100:.4f}%)')
print('\n5 tickers with most zero-volume days:')
(ohlcv.filter(F.col('volume') == 0)
      .groupBy('ticker').count()
      .orderBy(F.desc('count'))
      .limit(5).show(truncate=False))

Rows with volume == 0: 3,865 / 1,940,547 (0.1992%)

5 tickers with most zero-volume days:


+------+-----+
|ticker|count|
+------+-----+
|SW    |2409 |
|AMCR  |1371 |
|HWM   |35   |
|CHTR  |18   |
|VRT   |12   |
+------+-----+



## Daily-return outliers (features layer)

Scan computed returns for `|simple_ret| > 50%` — typically stock splits or one-off corporate actions on the raw `close` series (`adj_close` would absorb these; the existing `src/features.py:24` bug means we don't yet).

In [10]:
feats = spark.read.parquet('../data/parquet/features')
ret_total = feats.filter(F.col('simple_ret').isNotNull()).count()
outliers = feats.filter(F.abs(F.col('simple_ret')) > 0.5).count()
print(f'Daily |simple_ret| > 50%: {outliers:,} / {ret_total:,} '
      f'({outliers/ret_total*100:.4f}%)')
print('\n10 tickers with most return outliers (likely splits):')
(feats.filter(F.abs(F.col('simple_ret')) > 0.5)
      .groupBy('ticker').count()
      .orderBy(F.desc('count'))
      .limit(10).show(truncate=False))

Daily |simple_ret| > 50%: 16 / 1,940,044 (0.0008%)

10 tickers with most return outliers (likely splits):


+------+-----+
|ticker|count|
+------+-----+
|VRTX  |2    |
|PCG   |2    |
|HOOD  |1    |
|AMD   |1    |
|GL    |1    |
|BLDR  |1    |
|APA   |1    |
|TRGP  |1    |
|OXY   |1    |
|KDP   |1    |
+------+-----+



## Missing-data handling — where each choice lives

| Layer | Decision | Source |
|---|---|---|
| OHLCV ingestion | Drop rows missing `close` (the price the panel anchors on) | `src/download_ohlcv.py:60` |
| OHLCV ingestion | `volume.fillna(0)` — keep halt days but flag them | `src/download_ohlcv.py:66` |
| OHLCV ingestion | Per-batch retry × 2 with 60 s backoff; failures logged | `src/download_ohlcv.py:77-95` |
| FRED ingestion | Daily reindex + forward-fill (monthly series → daily panel) | `src/download_fred.py:58` |
| FRED ingestion | Drop leading rows where every series is still null | `src/download_fred.py:63` |
| Features | Window/lag functions emit `null` at series start; propagate | `src/features.py` (Spark default) |
| Features | Persisted partitioned by year for partition-pruned queries | `notebooks/02_features.ipynb` (`partitionBy('year')`) |
| Strategies | `dropna(subset=[lookback_col])` before quantile selection | `src/strategies.py:33`, `:83` |
| Backtest | `.fillna(0)` on weights/returns matrices — missing = no exposure | `src/backtest.py:32`, `:46` |
| Metrics | `dropna()` + `isnan` guards on every metric | `src/metrics.py:22-25` |
| Universe | Today's S&P 500 only (no point-in-time membership) | `src/universe.py` — survivorship caveat, see nb 06/07 |

In [11]:
spark.stop()